# NB5 · Güvenlik bariyerleri ve yönetişim

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---

Bu defter modelin başarımını iyileştirmez. Modelin ne zaman konuşmaması gerektiğini
belirler.

Tek bir kontrol hücresi bulunmaktadır. Bu aşamada denetim işi tamamen size geçmiştir;
defter yalnızca sonuçları görünür kılar.


## Hazırlık


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for modul in ['checks.py', 'evaluate.py', 'explain.py', 'safety.py',
              'mimic_web.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{modul}', modul)

import numpy as np
import pandas as pd
import checks, evaluate as ev, explain as ex, safety as sf

checks.LANG = ev.LANG = sf.LANG = 'tr'


In [ ]:
# Sabit hücre. Önceki defterlerin çıktısını yeniden kurar.
import pipeline as pl

durum = pl.prepare(verbose=False)
model = durum['model']
test = durum['test']
ozellikler = durum['features']
olasilik = durum['probabilities']
y_test = durum['y_test']
esik = ev.threshold_for_sensitivity(y_test, olasilik, target=0.80)
print(f'Model ve tahminler hazır. Çalışma eşiği: {esik:.3f}')


---

## Sabit bölüm · Çekimserlik bandı

Bant seçilmez, veriden türetilir. Bir çekimserlik bütçesi belirlenir ve bant o bütçeyi
dolduracak kadar genişletilir.

Asıl önemli olan denetimidir. Aşağıdaki çıktı bant içindeki doğrulukla bant dışındaki
doğruluğu karşılaştırır. İkisi birbirine yakınsa bant gerçek belirsizliği ayırmıyor
demektir ve görüntü olsun diye tutulmak yerine terk edilmelidir.


In [ ]:
# Sabit hücre. Bariyerleri kurup sistemi birleştirir.
bant = sf.choose_band(y_test, olasilik, esik, max_abstain=0.20)
politika = sf.AbstentionPolicy(esik, bant['band'])
kayma = sf.DriftDetector().fit(durum['train'][ozellikler])
dogrulayici = sf.PhysiologicalValidator().fit(durum['train'][ozellikler])
korumali = sf.GuardedModel(model, politika, kayma, dogrulayici)

for anahtar, deger in bant.items():
    print(f'{anahtar:<24} {deger}')


---

## Adım 1 · Eksik veri bariyeri

Aşağıdaki hücre bir sorunu göstermektedir. Bütün değerleri eksik olan bir hasta için
sistemin ne yaptığına bakınız.


In [ ]:
bos_hasta = test[ozellikler].iloc[[0]].copy()
bos_hasta.loc[bos_hasta.index[0], :] = np.nan
print('Bariyersiz karar:', korumali.predict_one(bos_hasta)['decision'])


Tamamlayıcı bütün boşlukları eğitim kümesinin ortancalarıyla doldurmakta ve model
kendinden emin bir olasılık üretmektedir. Hakkında hiçbir bilgi bulunmayan bir hasta
için sistem bir karar önermektedir. Hata iletisi çıkmaz, kod kusursuz çalışır.

Tamamlayıcı istendiğinde eklenir; ne zaman devre dışı kalması gerektiği ise sorulmadıkça
konuşulmaz. Bir sonraki istemde bu bariyeri kuracaksınız. Eşiğin kaç olacağı klinik bir
karardır: Bir hastanın kaç özniteliği eksikse tahmin üretilmemelidir?


### İstem 1

```
korumali adında bir nesne var. predict_one adında bir yöntemi bulunuyor; tek satırlık
bir DataFrame alıp sözlük döndürüyor.

Bunu saran tek bir Python fonksiyonu yaz. Fonksiyon, gelen satırdaki eksik öznitelik
oranını hesaplasın. Oran belirlediğim sınırı aşarsa tahmin üretmesin ve eksik veri
gerekçesiyle klinisyene devretsin. Aşmazsa korumali.predict_one çağrısının sonucunu
döndürsün.

Sınırı fonksiyonun dışında, adı büyük harflerle yazılmış bir sabit olarak tanımla ve
bunun teknik bir varsayılan değil klinik bir karar olduğunu yorum satırı olarak ekle.

KABUL ÖLÇÜTLERİ
tahmin_et adında bir fonksiyon üret. Tek satırlık bir DataFrame alsın.
Döndürdüğü sözlükte decision ve probability anahtarları bulunsun.
Bütün değerleri eksik bir satır verildiğinde probability değeri None olsun.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 1


In [ ]:
sonuc = tahmin_et(bos_hasta)
print('Bariyerli karar:', sonuc['decision'])
print('Olasılık       :', sonuc['probability'])
print()
normal = tahmin_et(test[ozellikler].iloc[[1]])
print('Olağan hasta   :', normal['decision'], '·', normal['probability'])


---

## Sabit bölüm · Kırmızı takım

Aşağıdaki hücre sistemi zorlayan beş girdi kurar ve her birinde ne yaptığını gösterir.
Girdilerden biri bariz biçimde bozuk değil, klinik olarak makul bir vakadır. Asıl önemli
olan o satırdır; yalnızca bariz bozuk girdilerde başarısız olan bir sistem gerçekte
sınanmamıştır.


In [ ]:
sf.red_team(korumali, test[ozellikler])


---

## Yönetişim belgeleri

Son adımda iki belge üretilecektir: Model kartı ve düzenleyici triyaj notu. Aşağıdaki
hücre, bu defterlerde ölçtüğünüz değerleri toplayıp isteme eklenecek bağlam bloğunu
hazırlar.


In [ ]:
rapor = ev.honest_report(y_test, olasilik,
                        groups=test['gender'] if 'gender' in test.columns else None,
                        target_sensitivity=0.80, label='bariyerli sistem')

baglam = f'''SİSTEM ÖZETİ
Amaç: Yoğun bakıma kabulden altı saat sonra üç günden uzun kalış riskinin öngörülmesi.
Veri: MIMIC-IV demo, tek merkez, {len(durum['cohort'])} yatış.
Model: Lojistik regresyon, sınıf ağırlıklı, hasta düzeyinde ayrım.

DEĞERLENDİRME
AUC {rapor['discrimination']['auc']:.3f} (95% GA {rapor['discrimination']['ci_low']:.3f}-{rapor['discrimination']['ci_high']:.3f})
Kalibrasyon eğimi {rapor['calibration']['slope']:.2f}
Eşik {rapor['threshold']:.3f}: duyarlılık {rapor['operating_point']['sensitivity']:.3f}, PKD {rapor['operating_point']['ppv']:.3f}
Yüz hastada {rapor['clinical']['alerts_fired']} uyarı, {rapor['clinical']['true_alerts']} doğru.

BARİYERLER
Çekimserlik bandı {bant['low']:.3f}-{bant['high']:.3f}, vakaların {bant['abstain_fraction']:.0%}'i.
Bant denetimi: {bant['verdict']}
Eksik veri bariyeri kuruldu.
'''
print(baglam)


### İstem 2

Yukarıdaki bağlam bloğunu kopyalayıp aşağıdaki istemin başına ekleyiniz. Bu istem kod
değil belge üretir.

```
Bu sistem için iki belge üret. Çıktıyı markdown olarak ver, kod üretme.

BİRİNCİ BELGE: Model kartı. Amaçlanan kullanım ve kullanıcılar; kapsam dışı
kullanımlar; eğitim verisi ve sınırlılıkları; alt grup dökümü dâhil değerlendirme
sonuçları; çalışma eşiği ve bu eşiği kimin seçtiği; bilinen başarısızlık biçimleri;
çekimserlik ve devretme davranışı; devreye alma sonrası izleme planı.

İKİNCİ BELGE: Düzenleyici triyaj notu. Şu dört soruyu cevapla:
- AB Tıbbi Cihaz Tüzüğü 2017/745 Kural 11 kapsamında bu yazılım tıbbi cihaz sayılır
  mı, hangi gerekçeyle?
- Cihaz sayılıyorsa AB Yapay Zekâ Yasası'nın hangi eki uygulanır ve geçerli uyum
  tarihi nedir? Temmuz 2026'da yürürlüğe giren AB 2026/1744 sayılı Dijital Omnibus
  düzenlemesinin bu tarihleri değiştirdiğini dikkate al.
- FD&C Act 520(o)(1)(E) maddesindeki klinik karar destek muafiyetinin dört ölçütünü
  karşılar mı? Bağımsız gözden geçirme ölçütünü ayrıca ele al.
- Sağlık verisinin özel nitelikli kişisel veri olması nedeniyle 6698 sayılı Kanun'dan
  hangi yükümlülükler doğar?

Dört maddenin her birinde kendi güven düzeyini ve bir hukukçunun kontrol etmesi
gereken noktayı belirt. Hiçbirini hukuki görüş olarak sunma.
```


## Üretilen belgelerin denetlenmesi

Düzenleyici triyaj notunu dikkatle okuyunuz. Bu alandaki uyum tarihleri Temmuz 2026'da
değişmiştir ve pek çok araç hâlâ eski takvimi döndürecek kadar yeni bir değişikliktir.
Araç eski tarihi verdiyse bu, kendinden emin icadın somut bir örneğidir ve atölyenin
geri kalanında aklınızda tutmanız gereken davranış biçimidir.

Belgelerdeki her atıf için DOI veya resmî belge numarası isteyiniz. Verilemiyorsa atıf
kaldırılmalıdır.

Üretilen model kartını depodaki `templates/tr/model-card.md` şablonuyla karşılaştırınız.
Şablonda bulunup belgede bulunmayan her başlık cevaplanmamış bir sorudur.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir
yoğun bakım popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
